# 🚀 Knatware Technology – LLM Fine-Tuning & Tuning Notebook

**Product of Knatware Technology**  
**Developed by:** Kayode Okosi – LLM Developer  

This is a production-ready, fully commented Google Colab notebook designed for fine-tuning and parameter-efficient tuning of Large Language Models (LLMs).

### What this notebook covers
- Environment setup & GPU verification
- Installing all required libraries
- Loading base models from Hugging Face
- Preparing datasets (instruction / chat format)
- Full fine-tuning vs Parameter-Efficient Fine-Tuning (PEFT / LoRA / QLoRA)
- Training configuration with clear compulsory vs optional parameters
- Evaluation, inference, and model saving / pushing to Hugging Face Hub
- Best practices and common pitfalls

---
**Important Notes**
- Run this notebook on a **GPU runtime** (Runtime → Change runtime type → GPU).
- Free Colab T4 is sufficient for 7B models with QLoRA.
- Always monitor VRAM usage (`!nvidia-smi`).
- Replace placeholder values (model name, dataset, HF token, etc.) before running.

## 1. Environment Setup & GPU Check

Always start by verifying that a GPU is available. Fine-tuning without a GPU is extremely slow and not recommended.

In [ ]:
# Check GPU availability and details
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Please change runtime type to GPU.")

## 2. Install Required Libraries

We install the core stack used in modern LLM fine-tuning:
- `transformers` – model loading & training
- `datasets` – efficient data loading
- `peft` – Parameter-Efficient Fine-Tuning (LoRA, QLoRA, etc.)
- `bitsandbytes` – 4-bit / 8-bit quantization (critical for QLoRA)
- `trl` – Transformer Reinforcement Learning (useful for SFTTrainer)
- `accelerate` – distributed / mixed precision training
- `huggingface_hub` – push models to the Hub

In [ ]:
# Install / upgrade required packages
# Note: On Colab these are usually already partially installed; we force upgrade for consistency.

!pip install -q --upgrade pip
!pip install -q transformers datasets peft bitsandbytes trl accelerate huggingface_hub sentencepiece protobuf

# Optional but recommended for better logging & monitoring
!pip install -q wandb tensorboard

print("✅ All packages installed successfully.")

## 3. Import Libraries & Configuration

Central place for all imports and global configuration flags.

In [ ]:
import os
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login, notebook_login

# Suppress excessive logging
logging.set_verbosity_info()

print("✅ Libraries imported successfully.")

## 4. Hugging Face Authentication (Compulsory if pushing models or using gated models)

**Compulsory if:**
- You want to push the fine-tuned model to the Hugging Face Hub
- You are using a gated model (e.g., Llama-2, Llama-3, Gemma, Mistral Instruct, etc.)

Get your token from: https://huggingface.co/settings/tokens

In [ ]:
# Option 1: Interactive login (recommended in Colab)
# notebook_login()

# Option 2: Hard-code token (less secure – use only for private notebooks)
# HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# login(token=HF_TOKEN)

# For this public notebook we leave it commented.
# Uncomment and run one of the options above when needed.
print("ℹ️  Hugging Face login is currently skipped. Uncomment the cell above if required.")

## 5. Model & Dataset Configuration (COMPULSORY PARAMETERS)

These are the **most important parameters** you must set correctly.

### Compulsory Parameters Explained:
| Parameter | Description | Example |
|-----------|-------------|--------|
| `model_name` | Base model ID from Hugging Face | `"mistralai/Mistral-7B-v0.1"` |
| `dataset_name` | Dataset ID or local path | `"yahma/alpaca-cleaned"` |
| `new_model` | Name you will give your fine-tuned model | `"knatware-mistral-7b-alpaca"` |
| `output_dir` | Local folder where checkpoints are saved | `"./results"` |

**Recommendation for Colab free tier:** Start with a 7B model + QLoRA.

In [ ]:
# ============================================================
# COMPULSORY CONFIGURATION – CHANGE THESE VALUES
# ============================================================

# Base model to fine-tune (must be a causal LM)
# Popular choices:
#   "mistralai/Mistral-7B-v0.1"
#   "meta-llama/Meta-Llama-3-8B"
#   "google/gemma-7b"
#   "Qwen/Qwen2-7B"
model_name = "mistralai/Mistral-7B-v0.1"          # ← COMPULSORY

# Dataset to use for supervised fine-tuning (SFT)
# Must contain instruction / response style data or chat format
dataset_name = "yahma/alpaca-cleaned"             # ← COMPULSORY

# Name of the final fine-tuned model (used when saving / pushing)
new_model = "knatware-mistral-7b-alpaca"          # ← COMPULSORY

# Local directory for checkpoints and final model
output_dir = "./knatware_results"                 # ← COMPULSORY

# ============================================================
# OPTIONAL BUT HIGHLY RECOMMENDED
# ============================================================

# Maximum sequence length (affects VRAM heavily)
max_seq_length = 512                              # 512 is safe on T4; try 1024/2048 on A100

# Whether to use 4-bit quantization (QLoRA) – strongly recommended on Colab
use_4bit = True

# LoRA rank (higher = more capacity but more VRAM)
lora_r = 16

# LoRA alpha (scaling factor, usually 2× rank)
lora_alpha = 32

# Dropout for LoRA layers
lora_dropout = 0.05

print(f"Model: {model_name}")
print(f"Dataset: {dataset_name}")
print(f"New model name: {new_model}")
print(f"Output dir: {output_dir}")

## 6. Load Tokenizer

The tokenizer must match the base model.  
We also set `pad_token` if it is missing (common with Llama / Mistral).

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,          # Required for some models (e.g., Qwen, Phi)
    use_fast=True                    # Fast tokenizer (Rust-based) is preferred
)

# Many base models do not have a pad token defined.
# Setting it to eos_token is the most common and safe practice for causal LMs.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Optional: set padding side (right is standard for causal LMs during training)
tokenizer.padding_side = "right"

print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

## 7. BitsAndBytes Configuration (QLoRA) – Highly Recommended

QLoRA (Quantized LoRA) loads the base model in 4-bit precision, dramatically reducing VRAM usage while still allowing high-quality fine-tuning.

### Key Compulsory Parameters for BitsAndBytesConfig:
- `load_in_4bit=True` → enables 4-bit quantization
- `bnb_4bit_quant_type="nf4"` → NormalFloat4 (best quality)
- `bnb_4bit_compute_dtype` → usually `torch.float16` or `torch.bfloat16`
- `bnb_4bit_use_double_quant=True` → nested quantization for extra memory savings

In [ ]:
# Compute dtype: float16 is safest on T4; bfloat16 is better on A100/H100
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,                       # COMPULSORY for QLoRA
    bnb_4bit_quant_type="nf4",                   # Recommended: nf4 (NormalFloat4)
    bnb_4bit_compute_dtype=compute_dtype,        # COMPULSORY – must match GPU capability
    bnb_4bit_use_double_quant=True,              # Optional but recommended (saves more VRAM)
)

print("BitsAndBytesConfig created:")
print(bnb_config)

## 8. Load Base Model with Quantization

We load the model in 4-bit mode and prepare it for k-bit training (required by PEFT).

In [ ]:
print(f"Loading model: {model_name} ...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config if use_4bit else None,
    device_map="auto",               # Automatically places layers on available devices
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)

# Prepare model for k-bit training (freezes base weights, enables gradient checkpointing, etc.)
model = prepare_model_for_kbit_training(model)

# Optional but recommended: enable gradient checkpointing to save VRAM
model.gradient_checkpointing_enable()

print("✅ Model loaded and prepared for training.")
print(f"Model device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'N/A'}")

## 9. LoRA / PEFT Configuration

Instead of updating all billions of parameters, LoRA injects small trainable rank-decomposition matrices into attention layers.

### Compulsory LoRA Parameters:
- `r` (rank) – dimension of the low-rank matrices (8, 16, 32, 64 are common)
- `lora_alpha` – scaling factor (usually 2 × r)
- `target_modules` – which modules to apply LoRA to (model-specific)
- `lora_dropout` – regularization
- `bias` – usually "none"
- `task_type` – "CAUSAL_LM" for language modeling

In [ ]:
# Target modules depend on the model architecture.
# For Mistral / Llama / most modern decoder-only models these are standard:
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

peft_config = LoraConfig(
    r=lora_r,                            # COMPULSORY – rank of update matrices
    lora_alpha=lora_alpha,               # COMPULSORY – scaling
    lora_dropout=lora_dropout,           # Recommended
    bias="none",                         # Usually "none"
    task_type="CAUSAL_LM",               # COMPULSORY for causal language models
    target_modules=target_modules,       # COMPULSORY – which layers to adapt
)

# Apply PEFT to the model
model = get_peft_model(model, peft_config)

# Print trainable parameters summary (very useful)
model.print_trainable_parameters()

print("✅ LoRA / PEFT configuration applied.")

## 10. Load & Prepare Dataset

We use the classic Alpaca-cleaned dataset as an example.  
You can replace it with any instruction-following or chat dataset.

**Key requirement:** The dataset must have a text field (or we create one) that the trainer will use.

In [ ]:
# Load dataset from Hugging Face Hub
dataset = load_dataset(dataset_name, split="train")

print(f"Dataset loaded. Number of examples: {len(dataset)}")
print("Sample example:")
print(dataset[0])

# ----------------------------------------------------------
# Formatting function – COMPULSORY
# The trainer expects a single text column (or we use formatting_func).
# For Alpaca-style data we create a simple prompt template.
# ----------------------------------------------------------
def formatting_prompts_func(example):
    """
    Converts instruction / input / output into a single training string.
    This format works well for most base models.
    """
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output = example.get("output", "")

    if input_text:
        text = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
    else:
        text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

    return {"text": text}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, remove_columns=dataset.column_names)

print("\nFormatted sample:")
print(dataset[0]["text"][:500] + "...")

## 11. Training Arguments (SFTTrainer)

This is where most hyper-parameters live.

### Compulsory / Critical Parameters:
| Parameter | Meaning | Recommended Starting Point |
|-----------|---------|---------------------------|
| `output_dir` | Where checkpoints are saved | `./results` |
| `num_train_epochs` | How many full passes over data | 1–3 |
| `per_device_train_batch_size` | Batch size per GPU | 2–4 on T4 |
| `gradient_accumulation_steps` | Effective batch size multiplier | 4–16 |
| `learning_rate` | Peak learning rate | 2e-4 for LoRA |
| `max_seq_length` | Truncation / packing length | 512 |
| `optim` | Optimizer | `paged_adamw_8bit` (memory efficient) |
| `fp16` / `bf16` | Mixed precision | `fp16=True` on T4 |
| `logging_steps` | How often to log | 10–25 |
| `save_steps` | Checkpoint frequency | 100–500 |

**Effective batch size** = `per_device_train_batch_size` × `gradient_accumulation_steps` × number of GPUs

In [ ]:
training_arguments = TrainingArguments(
    output_dir=output_dir,                       # COMPULSORY
    num_train_epochs=1,                          # Start with 1; increase if needed
    per_device_train_batch_size=2,                # Reduce if OOM (Out of Memory)
    gradient_accumulation_steps=4,               # Effective batch size = 2 * 4 = 8
    optim="paged_adamw_8bit",                    # Memory-efficient optimizer (QLoRA)
    save_steps=100,                              # Save checkpoint every N steps
    logging_steps=25,                            # Log every N steps
    learning_rate=2e-4,                          # Standard for LoRA / QLoRA
    weight_decay=0.001,
    fp16=True,                                   # Use fp16 on T4 / most GPUs
    bf16=False,                                  # Set True only on A100/H100
    max_grad_norm=0.3,                           # Gradient clipping
    max_steps=-1,                                # -1 = use num_train_epochs
    warmup_ratio=0.03,                           # Warmup percentage of total steps
    group_by_length=True,                        # Groups similar length sequences → faster
    lr_scheduler_type="cosine",                  # Cosine schedule is popular
    report_to="none",                            # Change to "wandb" or "tensorboard" if desired
    # save_total_limit=3,                        # Keep only last N checkpoints (optional)
)

print("✅ TrainingArguments configured.")

## 12. Initialize SFTTrainer

`SFTTrainer` from the `trl` library simplifies supervised fine-tuning.

### Compulsory arguments for SFTTrainer:
- `model`
- `train_dataset`
- `peft_config` (or already applied)
- `tokenizer`
- `args` (TrainingArguments)
- `max_seq_length`
- `formatting_func` or a dataset that already has a `text` column

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,                     # Already applied, but passing is fine
    dataset_text_field="text",                   # Name of the column containing the training text
    max_seq_length=max_seq_length,               # COMPULSORY – truncation length
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,                               # Set True to pack multiple short examples (faster)
)

print("✅ SFTTrainer initialized. Ready to train.")

## 13. Start Training

This cell will take time depending on dataset size, model size, and hardware.

**Tips:**
- Watch the loss curve – it should steadily decrease.
- If you hit CUDA Out-Of-Memory, reduce `per_device_train_batch_size` or `max_seq_length`.
- You can interrupt and resume from the last checkpoint.

In [ ]:
# Start fine-tuning
print("🚀 Starting training...")
trainer.train()

print("✅ Training completed!")

## 14. Save the Fine-Tuned Adapter

With PEFT / LoRA we only save the small adapter weights (a few hundred MB), not the full model.

In [ ]:
# Save only the LoRA adapter
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

print(f"✅ Adapter and tokenizer saved to: {new_model}")

# Optional: also save the full TrainingArguments and config
trainer.save_model(new_model)

## 15. (Optional) Merge LoRA Adapter with Base Model

If you want a single standalone model file (useful for deployment), merge the adapter back into the base model.

**Warning:** Merging requires loading the full-precision base model → needs more VRAM.

In [ ]:
# Uncomment the block below if you want to merge.

# from peft import PeftModel

# # Reload base model in full precision (or fp16)
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     trust_remote_code=True,
# )

# # Load the adapter on top
# model = PeftModel.from_pretrained(base_model, new_model)

# # Merge and unload
# model = model.merge_and_unload()

# # Save the merged model
# model.save_pretrained("merged_" + new_model)
# tokenizer.save_pretrained("merged_" + new_model)

# print("✅ Merged model saved.")

print("ℹ️  Merge step is commented out. Uncomment if needed.")

## 16. Inference / Testing the Fine-Tuned Model

Quick generation test to verify the model learned something useful.

In [ ]:
# Simple generation pipeline using the trained model (still in memory)
prompt = """### Instruction:
Explain what LoRA is in simple terms.

### Response:
"""

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

## 17. Push to Hugging Face Hub (Optional)

Requires a valid Hugging Face token and that you have logged in (Section 4).

In [ ]:
# Uncomment after logging in

# trainer.model.push_to_hub(new_model, private=False)   # or private=True
# tokenizer.push_to_hub(new_model)

# print(f"✅ Model pushed to: https://huggingface.co/{new_model}")

print("ℹ️  Push-to-Hub is commented out. Uncomment after authentication.")

## 18. Best Practices & Troubleshooting Summary

### Memory (OOM) issues
- Lower `per_device_train_batch_size`
- Increase `gradient_accumulation_steps`
- Reduce `max_seq_length`
- Enable `gradient_checkpointing`
- Use QLoRA (`load_in_4bit=True`)

### Slow training
- Enable `packing=True` in SFTTrainer (if sequences are short)
- Use `group_by_length=True`
- Ensure you are on a GPU runtime

### Loss not decreasing
- Learning rate too high / too low → try 1e-4 to 3e-4
- Dataset quality / formatting issues
- Insufficient training steps / epochs

### Model collapses or produces garbage
- Check that `pad_token` is set correctly
- Verify the prompt template matches what the model expects
- Make sure you are not training on the EOS token incorrectly

---

**End of Notebook**  
Product of **Knatware Technology**  
Developed by **Kayode Okosi** – LLM Developer  

Feel free to adapt this notebook for your own datasets, models, and research.  
Happy fine-tuning! 🚀